# Exercise 4: Advanced Analytics

**Learning Objectives:**
- Window Functions: ROW_NUMBER, RANK, SUM OVER
- Running Totals and Moving Averages
- CTEs (Common Table Expressions)
- Data Visualization with Plotly

**Estimated Time:** 45-60 minutes

**Prerequisites:** Exercise 1-3 completed

In [ ]:
import duckdb
import pandas as pd

# Plotly for visualization
try:
    import plotly.express as px
except ImportError:
    %pip install plotly -q
    import plotly.express as px

conn = duckdb.connect(':memory:')

## Setup: Load Data

In [ ]:
# Load all required tables
conn.execute("""
    CREATE TABLE products AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv');
    CREATE TABLE product_categories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv');
    CREATE TABLE product_subcategories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv');
    CREATE TABLE sales_orders AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv');
    CREATE TABLE sales_details AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv');
    CREATE TABLE territories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv');
""")

print("✓ All tables loaded")

---
## Task 1: Window Functions - Ranking

### 1a) Rank products by price

Create a **ranking of all products by ListPrice** within each category.

**Expected Columns:**
- `category_name`
- `product_name`
- `list_price`
- `price_rank` (1 = most expensive)

**Syntax:**
```sql
SELECT 
    c.Name AS category_name,
    p.Name AS product_name,
    p.ListPrice AS list_price,
    ROW_NUMBER() OVER (
        PARTITION BY c.ProductCategoryID 
        ORDER BY p.ListPrice DESC
    ) AS price_rank
FROM products p
JOIN product_subcategories s ON p.ProductSubcategoryID = s.ProductSubcategoryID
JOIN product_categories c ON s.ProductCategoryID = c.ProductCategoryID
WHERE p.ListPrice > 0
```

**Show the first 20 results.**

In [ ]:
# Your code here:


### 1b) Top 3 per category only

Show only the **3 most expensive products per category**.

**Tip:** Use the query from 1a) as a subquery and filter with `WHERE price_rank <= 3`

**Syntax:**
```sql
SELECT * FROM (
    -- Query from 1a here
) ranked
WHERE price_rank <= 3
ORDER BY category_name, price_rank
```

In [ ]:
# Your code here:


---
## Task 2: Running Totals

Calculate the **running sum of sales** (TotalDue) over time for each territory.

**Tables:** `sales_orders` JOIN `territories`

**Expected Columns:**
- `territory_name`
- `order_date` (date only, no time)
- `daily_revenue` (TotalDue on this day)
- `running_total` (sum up to this day)

**Syntax:**
```sql
SELECT 
    t.Name AS territory_name,
    CAST(s.OrderDate AS DATE) AS order_date,
    SUM(s.TotalDue) AS daily_revenue,
    SUM(SUM(s.TotalDue)) OVER (
        PARTITION BY t.TerritoryID 
        ORDER BY CAST(s.OrderDate AS DATE)
    ) AS running_total
FROM sales_orders s
JOIN territories t ON s.TerritoryID = t.TerritoryID
GROUP BY t.TerritoryID, t.Name, CAST(s.OrderDate AS DATE)
ORDER BY territory_name, order_date
```

**Show the first 30 results.**

In [ ]:
# Your code here:


---
## Task 3: Moving Average

Calculate the **7-day moving average** of daily sales.

**Steps:**
1. First aggregate daily sales
2. Then calculate the moving average

**Syntax for Moving Average:**
```sql
AVG(daily_revenue) OVER (
    ORDER BY order_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
) AS moving_avg_7d
```

**Complete Query:**
```sql
WITH daily_sales AS (
    SELECT 
        CAST(OrderDate AS DATE) AS order_date,
        SUM(TotalDue) AS daily_revenue
    FROM sales_orders
    GROUP BY CAST(OrderDate AS DATE)
)
SELECT 
    order_date,
    daily_revenue,
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_7d
FROM daily_sales
ORDER BY order_date
```

In [ ]:
# Your code here:


---
## Task 4: CTEs (Common Table Expressions)

Create a **multi-step analysis** with CTEs:

1. **CTE 1: `sales_with_products`**
   - JOIN: sales_details + products + product_subcategories + product_categories
   - Columns: SalesOrderID, ProductID, product_name, category_name, LineTotal

2. **CTE 2: `category_sales`**
   - Aggregate by category_name
   - Columns: category_name, total_revenue

3. **Final Query:**
   - Calculate the **percentage** of each category of total revenue
   - Columns: category_name, total_revenue, percentage

**Syntax:**
```sql
WITH 
sales_with_products AS (
    SELECT 
        sd.SalesOrderID,
        sd.ProductID,
        p.Name AS product_name,
        c.Name AS category_name,
        sd.LineTotal
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories s ON p.ProductSubcategoryID = s.ProductSubcategoryID
    JOIN product_categories c ON s.ProductCategoryID = c.ProductCategoryID
),
category_sales AS (
    SELECT 
        category_name,
        SUM(LineTotal) AS total_revenue
    FROM sales_with_products
    GROUP BY category_name
)
SELECT 
    category_name,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(100.0 * total_revenue / SUM(total_revenue) OVER (), 2) AS percentage
FROM category_sales
ORDER BY total_revenue DESC
```

In [ ]:
# Your code here:


---
## Task 5: Visualization

### 5a) Bar Chart: Top 10 Products by Revenue

1. Write a query for the top 10 products by revenue (LineTotal)
2. Save the result as DataFrame
3. Create a bar chart with Plotly

**Query:**
```sql
SELECT 
    p.Name AS product_name,
    ROUND(SUM(sd.LineTotal), 2) AS total_revenue
FROM sales_details sd
JOIN products p ON sd.ProductID = p.ProductID
GROUP BY p.ProductID, p.Name
ORDER BY total_revenue DESC
LIMIT 10
```

**Plotly:**
```python
df = conn.execute("...").df()
fig = px.bar(df, x='product_name', y='total_revenue', title='Top 10 Products by Revenue')
fig.show()
```

In [ ]:
# Your code here:


### 5b) Line Chart: Monthly Sales Trends

1. Aggregate sales by year-month
2. Create a line chart

**Query:**
```sql
SELECT 
    strftime(OrderDate, '%Y-%m') AS month,
    ROUND(SUM(TotalDue), 2) AS monthly_revenue
FROM sales_orders
GROUP BY strftime(OrderDate, '%Y-%m')
ORDER BY month
```

**Plotly:**
```python
fig = px.line(df, x='month', y='monthly_revenue', title='Monthly Revenue Trend')
fig.show()
```

In [ ]:
# Your code here:


---
## Task 6: Pivot Table

Create a **pivot table** with:
- **Rows:** Product categories
- **Columns:** Years (2011, 2012, 2013, 2014)
- **Values:** Total revenue

**Syntax with CASE WHEN:**
```sql
SELECT 
    c.Name AS category,
    ROUND(SUM(CASE WHEN YEAR(so.OrderDate) = 2011 THEN sd.LineTotal ELSE 0 END), 0) AS "2011",
    ROUND(SUM(CASE WHEN YEAR(so.OrderDate) = 2012 THEN sd.LineTotal ELSE 0 END), 0) AS "2012",
    ROUND(SUM(CASE WHEN YEAR(so.OrderDate) = 2013 THEN sd.LineTotal ELSE 0 END), 0) AS "2013",
    ROUND(SUM(CASE WHEN YEAR(so.OrderDate) = 2014 THEN sd.LineTotal ELSE 0 END), 0) AS "2014"
FROM sales_details sd
JOIN products p ON sd.ProductID = p.ProductID
JOIN product_subcategories s ON p.ProductSubcategoryID = s.ProductSubcategoryID
JOIN product_categories c ON s.ProductCategoryID = c.ProductCategoryID
JOIN sales_orders so ON sd.SalesOrderID = so.SalesOrderID
GROUP BY c.Name
ORDER BY c.Name
```

In [ ]:
# Your code here:


---
## Bonus Task ⭐⭐⭐: RFM Analysis

Perform an **RFM Analysis** (Recency, Frequency, Monetary):

- **Recency:** How many days since the last order?
- **Frequency:** How many orders has the customer placed?
- **Monetary:** How much has the customer spent in total?

Segment customers into **5 groups** (1=worst, 5=best) for each dimension.

**Syntax:**
```sql
WITH customer_rfm AS (
    SELECT 
        CustomerID,
        DATEDIFF('day', MAX(OrderDate), '2014-06-30') AS recency,
        COUNT(*) AS frequency,
        SUM(TotalDue) AS monetary
    FROM sales_orders
    GROUP BY CustomerID
),
rfm_scores AS (
    SELECT 
        CustomerID,
        recency,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency) AS f_score,
        NTILE(5) OVER (ORDER BY monetary) AS m_score
    FROM customer_rfm
)
SELECT 
    CustomerID,
    recency,
    frequency,
    ROUND(monetary, 2) AS monetary,
    r_score,
    f_score,
    m_score,
    r_score + f_score + m_score AS total_score
FROM rfm_scores
ORDER BY total_score DESC
LIMIT 20
```

In [ ]:
# Your code here:


---
## 🎉 Congratulations!

You now master Advanced Analytics with DuckDB!

**What you learned:**
- ✅ Window Functions: ROW_NUMBER, RANK, NTILE
- ✅ Running Totals with SUM() OVER
- ✅ Moving Averages with ROWS BETWEEN
- ✅ CTEs for multi-step analyses
- ✅ Pivot Tables with CASE WHEN
- ✅ Data Visualization with Plotly